In [6]:
def f(x):
    return x ** 3 - 2 * x

def numerical_derivative(f, x, eps=1e-5):
    return (f(x + eps) - f(x - eps)) / ( 2 * eps)

print(numerical_derivative(f, 1))
print(3 * 1 ** 2 - 2)

1.000000000095369
1


In [1]:
from llm.attention import casual_mask

print(casual_mask(4).data)

[[ 0.e+00 -1.e+09 -1.e+09 -1.e+09]
 [ 0.e+00  0.e+00 -1.e+09 -1.e+09]
 [ 0.e+00  0.e+00  0.e+00 -1.e+09]
 [ 0.e+00  0.e+00  0.e+00  0.e+00]]


In [1]:
import numpy as np
from llm.tensor import Tensor
from llm.nn import manual_seed
from llm.attention import CasualSelfAttention

manual_seed(0)

attn = CasualSelfAttention(n_embd=8, n_head=2)
x = np.random.default_rng(1).normal(size=(2, 5, 8)).astype(np.float32)
y1 = attn(Tensor(x)).data

print(y1.shape)

x2 = x.copy()
x2[:, -1] +=  1.0
y2 = attn(Tensor(x2)).data
print(np.abs(y2 - y1).max(axis=(0, 2)))

(2, 5, 8)
[0.         0.         0.         0.         0.00153796]


In [5]:
import numpy as np
from llm.tensor import Tensor, numerical_grad
from llm.nn import manual_seed
from llm.attention import CausalSelfAttention

manual_seed(0)
attn = CausalSelfAttention(n_embd=6, n_head=2)
rng = np.random.default_rng(1)

for p in attn.parameters():
    p.data = rng.normal(size=p.shape) * 0.5   # float64
x = Tensor(rng.normal(size=(2, 4, 6)))
G = rng.normal(size=(2, 4, 6))

loss = lambda: float((attn(x).data * G).sum())
(attn(x) * Tensor(G)).sum().backward()

for i, p in enumerate(attn.parameters()):
    err = np.abs(p.grad - numerical_grad(loss, p.data)).max()
    biggest = np.abs(p.grad).max()
    print(i, p.shape, f"error {err:.1e}   largest grad {biggest:.1e}")

0 (6, 6) error 1.8e-09   largest grad 6.2e+00
1 (6,) error 5.5e-10   largest grad 2.3e+00
2 (6, 6) error 2.1e-09   largest grad 1.9e+00
3 (6,) error 8.9e-10   largest grad 8.6e-16
4 (6, 6) error 2.0e-09   largest grad 5.5e+00
5 (6,) error 2.3e-09   largest grad 2.9e+00
6 (6, 6) error 1.1e-09   largest grad 1.1e+01
7 (6,) error 6.7e-10   largest grad 3.8e+00
